# Revision analysis — PLOS Computational Biology, round 2

This notebook builds the new supplementary figure(s) requested by reviewers, following the same structure as the other notebooks in this repo (R via IRkernel). See project notes for the full reviewer-comment mapping.

Planned panels (may end up as one figure or split into two, decided once results are in):

- **Panel A** — CLIMB component ablation, merged across the in-vitro mixtures *and* all 10 real cross-dataset pairs used elsewhere in the paper, plus the manuscript's own NNLS baseline for comparison. Answers Reviewer 1 W1 (both the iterative-reweighting and cell-level-resolution halves) and doubles as a reproduction of the manuscript's own SFig. 3E-F claim.
- **Panel B** — Reference completeness/coverage sweep on the COVID-19 nasal-swab dataset (non-hematopoietic), same three methods as Panel A, tracked across reference depth (5 to 500 cells/type). Answers Reviewer 1 W5 (limitations re: scRNA-seq reference bias, specifically in non-hematopoietic contexts) and further supports W1 (NNLS is coverage-insensitive; CLIMB is not).
- **Panel C** — CLIFF identifiability under correlated/low-variance cell-subtype proportions across samples — Reviewer 2, comment #1.
- **Panel D** — EM convergence / initialization sensitivity: restart scatter across rho — Reviewer 2, comment #2.
- **Panel E** — EM convergence / initialization sensitivity: initialization-landscape heatmap — Reviewer 2, comment #2 (continued).
- **Panel F** — Runtime / computational complexity — Reviewer 2, comment #3.
- **Panel G** — Uncertainty propagation from CLIMB proportions into CLIFF sensitivities — Reviewer 2, comment #4.

Both Panels A and B use the same three-way method comparison, with a consistent naming/color scheme defined once in Setup:
- **Cell-type aggregated (NNLS)** — the manuscript's "prototypical" baseline (Methods Eq. 1): non-negative least squares against the *averaged* cell-type expression profile.
- **Single-cell, uncorrected (props.init)** — CLIMB's first-pass fit on the full single-cell matrix, before iterative correction.
- **Single-cell, empirical-Bayes corrected (props.corrected)** — CLIMB's default output, after the iterative empirical-Bayes reweighting step.

## Setup

In [ ]:
suppressMessages({
    library(Biobase)
    library(ClimbTheCliff)
    library(glmnet)
    library(ggplot2)
    library(nnls)
    library(cowplot)
    library(reshape2)  # cliff() calls dcast() internally; not auto-attached by the package
    library(MASS)      # mvrnorm(), for Panel C's correlated-proportion simulation
    library(ggnewscale) # lets Panel G use two independent color scales in one plot
})
set.seed(1)

# Shared naming/color scheme for the 3-way method comparison used in Panels A and B.
# grepl-based (not exact match) so it's robust to any descriptive suffix on the raw label
# (e.g. "props.init (single-pass)" as well as plain "props.init").
method_levels <- c("Cell-type aggregated (NNLS)",
                    "Single-cell, uncorrected (props.init)",
                    "Single-cell, empirical-Bayes corrected (props.corrected)")
relabel_method <- function(x) {
    x <- as.character(x)
    out <- rep(NA_character_, length(x))
    out[grepl("^NNLS", x)] <- method_levels[1]
    out[grepl("^props\\.init", x)] <- method_levels[2]
    out[grepl("^props\\.corrected", x)] <- method_levels[3]
    factor(out, levels = method_levels)
}
method_palette <- RColorBrewer::brewer.pal(3, "Dark2")
names(method_palette) <- method_levels

rmse <- function(pred, truth) sqrt(mean((pred - truth)^2))
per_sample_rmse <- function(pred_mat, truth_mat) sqrt(rowMeans((pred_mat - truth_mat)^2))
se <- function(x) sd(x) / sqrt(length(x))

# Cell-type label cleanup used by climb() internally (strips characters like "-", " ", "+"
# from cell-type names) — needed when loading a dataset directly by us (e.g. Panel B's
# COVID-19 reference) so labels match what climb()/true_prop use downstream, mirroring how
# the original repo notebooks pre-clean labels before ever calling climb().
reformat_celltype_labels <- function(x) {
    x <- gsub("\\-$", "", as.vector(x))
    x <- gsub("\\+", "", x)
    x <- gsub("minus", "", x)
    x <- gsub("plus", "", x)
    x <- gsub("\\ ", "\\.", x)
    x <- gsub("[^[:alnum:] ]", "", x)
    factor(x)
}

# Synthetic bulk/single-cell generator, shared across Panels C-F (Reviewer 2's four
# comments): controllable n_samples, n_celltypes (k), n_genes (g), and reference depth.
# Ground truth (true_props) is known exactly, unlike any real dataset.
make_synthetic <- function(n_samples, k, g, n_cells_per_type = 200, seed = NULL) {
    if (!is.null(seed)) set.seed(seed)
    celltypes <- paste0("CT", 1:k)
    sc_cellType <- rep(celltypes, each = n_cells_per_type)
    n_cells <- length(sc_cellType)
    profile <- matrix(rgamma(g * k, shape = 2, rate = 1), nrow = g, ncol = k)
    sc_mat <- matrix(0, nrow = g, ncol = n_cells)
    for (j in seq_len(n_cells)) {
        kk <- match(sc_cellType[j], celltypes)
        sc_mat[, j] <- rpois(g, lambda = profile[, kk] * 200)
    }
    rownames(sc_mat) <- paste0("gene", 1:g); colnames(sc_mat) <- paste0("cell", 1:n_cells)
    sc <- ExpressionSet(assayData = sc_mat); sc$cellType <- sc_cellType

    true_props <- matrix(runif(n_samples * k), nrow = n_samples, ncol = k)
    true_props <- true_props / rowSums(true_props)
    bulk_mat <- matrix(0, nrow = g, ncol = n_samples)
    for (i in seq_len(n_samples)) {
        lambda_i <- profile %*% true_props[i, ] * 200
        bulk_mat[, i] <- rpois(g, lambda = lambda_i)
    }
    rownames(bulk_mat) <- paste0("gene", 1:g); colnames(bulk_mat) <- paste0("sample", 1:n_samples)
    bulk <- ExpressionSet(assayData = bulk_mat)
    drug_data <- data.frame(sample = colnames(bulk),
        auc = pmin(pmax(rowSums(true_props * matrix(runif(k, 0.2, 0.8), n_samples, k, byrow = TRUE)) +
                           rnorm(n_samples, sd = 0.05), 0.01), 0.99))
    list(sc = sc, bulk = bulk, true_props = true_props, drug_data = drug_data)
}

## Panel A — CLIMB component ablation, merged across in-vitro + 10 cross-dataset pairs (Reviewer 1, W1)

`climb()` returns both `props.init` (first-pass fit, before the empirical-Bayes iterative correction) and `props.corrected` (after it). Comparing the two isolates the iterative-reweighting component. We also include the manuscript's own precomputed NNLS baseline (`nnls_prop.csv`, already present for every dataset in this repo) to isolate the cell-level-resolution component — NNLS uses the *averaged* cell-type profile instead of the full single-cell matrix.

This panel merges two sources of real ground truth:
1. The 6 in-vitro bulk mixtures (proportions known from precise cell counting).
2. All 10 real cross-dataset pseudo-bulk pairs used elsewhere in the paper (AML, Breast, CRC, GBM, Melanoma, each in both directions), using the already-computed `climb_prop_init_3pass.csv` / `climb_prop_corrected_3pass.csv` / `true_prop.csv` per pair — this reproduces the manuscript's own existing claim (p.7, SFig. 3E-F): *"CLIMB-corrected demonstrated enhanced accuracy and resilience to bias over CLIMB-init in most cross-dataset pseudo-bulk analysis."*

No new simulation needed for #2 — it's a recombination of existing per-pair results into one consistent view, side by side with the in-vitro result.

In [ ]:
sc.es <- readRDS("data/invitro_experiment/invitro_sc_es.RDS")
bulk.es <- readRDS("data/invitro_experiment/invitro_bulk_es.RDS")
true_prop <- as.matrix(read.csv("data/invitro_experiment/true_prop.csv", row.names = 1))
sc.es$cellType <- factor(sc.es$cellType)
cn <- colnames(true_prop)

climb_out <- climb(sc.es, bulk.es, mode = "abundance", verbose = TRUE)
nnls_prop_invitro <- read.csv("data/invitro_experiment/tables/nnls_prop.csv", row.names = "X")

# Reusable direct NNLS implementation (Methods Eq. 1: x_n ~ S . w_n against the averaged
# cell-type profile S) — needed later for Panel B's coverage sweep, where NNLS has to be
# recomputed at each subsampled reference size rather than loaded from a precomputed file.
nnls_deconv <- function(sc, bulk) {
    common_genes <- intersect(rownames(bulk), rownames(sc))
    sc_mat <- exprs(sc)[common_genes, ]
    S <- sapply(levels(sc$cellType), function(ct) rowMeans(sc_mat[, sc$cellType == ct, drop = FALSE]))
    bulk_mat <- exprs(bulk)[common_genes, ]
    props <- t(apply(bulk_mat, 2, function(xn) {
        w <- coef(nnls::nnls(S, xn))
        w / sum(w)
    }))
    colnames(props) <- colnames(S)
    props
}

df_invitro <- data.frame(
    pair = "invitro-mixture", sample = rep(rownames(true_prop), 3),
    method = rep(c("props.init", "props.corrected", "NNLS"), each = nrow(true_prop)),
    rmse = c(per_sample_rmse(as.matrix(climb_out$props.init)[, cn], true_prop),
             per_sample_rmse(as.matrix(climb_out$props.corrected)[, cn], true_prop),
             per_sample_rmse(as.matrix(nnls_prop_invitro)[, cn], true_prop))
)
df_invitro

In [ ]:
cross_pairs <- c("AML_na2vg", "AML_vg2na", "GBM_10x2ss2", "GBM_SS2tenX",
                  "CRC_kha2lee", "CRC_lee2kha", "BREAST_gray2wu", "BREAST_wu2gray",
                  "MEL_jer2tir", "MEL_tir2jer")

rows <- list()
for (p in cross_pairs) {
    d <- file.path("data/pseudobulks_climb", p)
    true_p <- read.csv(file.path(d, "true_prop.csv"), row.names = "X")
    init_p <- read.csv(file.path(d, "climb_prop_init_3pass.csv"), row.names = "X")
    corr_p <- read.csv(file.path(d, "climb_prop_corrected_3pass.csv"), row.names = "X")
    nnls_p <- read.csv(file.path(d, "nnls_prop.csv"), row.names = "X")
    cn_ <- Reduce(intersect, list(colnames(true_p), colnames(init_p), colnames(corr_p), colnames(nnls_p)))
    rows[[length(rows) + 1]] <- data.frame(pair = p, sample = rownames(true_p), method = "props.init",
        rmse = per_sample_rmse(init_p[, cn_], true_p[, cn_]))
    rows[[length(rows) + 1]] <- data.frame(pair = p, sample = rownames(true_p), method = "props.corrected",
        rmse = per_sample_rmse(corr_p[, cn_], true_p[, cn_]))
    rows[[length(rows) + 1]] <- data.frame(pair = p, sample = rownames(true_p), method = "NNLS",
        rmse = per_sample_rmse(nnls_p[, cn_], true_p[, cn_]))
}
df_crossdataset <- do.call(rbind, rows)

df_panelA <- rbind(df_invitro, df_crossdataset)
df_panelA$method <- relabel_method(df_panelA$method)
df_panelA$pair <- factor(df_panelA$pair, levels = c("invitro-mixture", cross_pairs))

summ_A <- aggregate(rmse ~ pair + method, df_panelA, function(x) c(mean = mean(x), se = se(x)))
summ_A <- do.call(data.frame, summ_A)
names(summ_A) <- c("pair", "method", "mean", "se")

g_A <- ggplot(summ_A, aes(x = pair, y = mean, color = method)) +
    geom_pointrange(aes(ymin = mean - se, ymax = mean + se),
                     position = position_dodge(0.5), size = 0.5, fatten = 1.2) +
    theme_classic(base_size = 12) + xlab(NULL) + ylab("CLIMB per-sample RMSE") +
    theme(axis.text.x = element_text(angle = 35, hjust = 1)) +
    scale_color_manual(values = method_palette)
g_A

out_dir_A <- "../../results/panelA"
dir.create(out_dir_A, recursive = TRUE, showWarnings = FALSE)
write.csv(df_panelA, file.path(out_dir_A, "panelA_merged_data.csv"), row.names = FALSE)
ggsave(file.path(out_dir_A, "panelA_merged_ablation.pdf"), g_A + theme(legend.position = "none"),
       width = 6.5, height = 3.2)

## Panel B — Reference completeness/coverage sweep on COVID-19 nasal-swab data (Reviewer 1, W5 and W1)

**Note on data source:** `data/simulation_aml/celltype_expr/*_{5,10,...,100}each.RDS` was initially assumed to be a reusable reference-coverage sweep, but inspection showed it's something else entirely — an older, unused differential-expression sample-size sweep (WT vs over-expression conditions; `climb_out_5each.RDS` has an old schema with `props`, `props.ba`, `expr.pred`, `DE.expr.conditions`, and `bulk_sim_{N}each.RDS` has `2*N` samples, not `N` reference cells per type). It isn't referenced by any current notebook.

**Why COVID-19 nasal-swab instead of in-vitro for this panel:** first tried on the in-vitro reference — the coverage-degradation trend was clear (RMSE 0.28 → 0.04), but NNLS came out 4-5x worse than CLIMB throughout, an unrealistically large, dataset-specific gap that dominated the plot. The COVID-19 nasal-swab pseudo-bulk pair (`data/pseudobulks_climb/COVID19_nasalSwab_sc_es.RDS` / `..._pbulk_es.RDS`, 18 respiratory epithelial/immune cell types) gives a much more comparable NNLS baseline *and* is genuinely non-hematopoietic, directly matching W5's wording. At 5-100 cells/type the coverage trend looked flat, but extending to 200-500 cells/type revealed a real, monotonic improvement for `props.corrected` — smaller in magnitude than the in-vitro result, but real.

Approach: subsample the COVID-19 single-cell reference down to a fixed number of cells per cell type (5 to 500; the rarest type, Mast Cells, only has 6 total), rerun all three methods (same naming/color scheme as Panel A) on the real bulk mixtures, 5 random subsamples per coverage level.

This directly supports **W1**'s cell-resolution question too: NNLS is structurally incapable of using more reference cells (it only ever sees the mean), so if it stays flat while CLIMB improves, that's a coverage-matched demonstration that single-cell resolution is what lets CLIMB benefit from more reference data at all.

In [ ]:
sc.covid <- readRDS("data/pseudobulks_climb/COVID19_nasalSwab_sc_es.RDS")
bulk.covid <- readRDS("data/pseudobulks_climb/COVID19_nasalSwab_pbulk_es.RDS")
sc.covid$cellType <- reformat_celltype_labels(sc.covid$cellType)
true_prop_covid <- as.matrix(pData(bulk.covid))
colnames(true_prop_covid) <- gsub(".", "", colnames(true_prop_covid), fixed = TRUE)
cn_covid <- colnames(true_prop_covid)

levels_n <- c(5, 25, 100, 200, 500)
n_reps <- 5

rows <- list()
for (n_per_type in levels_n) {
    for (rep in 1:n_reps) {
        set.seed(rep * 1000 + n_per_type)
        idx <- unlist(lapply(levels(sc.covid$cellType), function(ct) {
            cells <- which(sc.covid$cellType == ct)
            sample(cells, min(n_per_type, length(cells)), replace = FALSE)
        }))
        sc_sub <- sc.covid[, idx]
        res_sub <- climb(sc_sub, bulk.covid, mode = "abundance", verbose = FALSE,
                          min.n.cells = min(5, n_per_type))
        nnls_prop_sub <- nnls_deconv(sc_sub, bulk.covid)

        method_preds <- list(
            "props.init" = as.matrix(res_sub$props.init)[, cn_covid],
            "props.corrected" = as.matrix(res_sub$props.corrected)[, cn_covid],
            "NNLS" = nnls_prop_sub[, cn_covid]
        )
        for (method_name in names(method_preds)) {
            rows[[length(rows) + 1]] <- data.frame(
                n_per_type = n_per_type, rep = rep, method = method_name,
                sample = rownames(true_prop_covid),
                rmse = per_sample_rmse(method_preds[[method_name]], true_prop_covid)
            )
        }
    }
}
df_coverage <- do.call(rbind, rows)
aggregate(rmse ~ n_per_type + method, df_coverage, function(x) c(mean = mean(x), sd = sd(x)))

In [ ]:
df_coverage$method <- relabel_method(df_coverage$method)
df_coverage$n_per_type_f <- factor(df_coverage$n_per_type, levels = levels_n)

summ_B <- aggregate(rmse ~ n_per_type_f + method, df_coverage, function(x) c(mean = mean(x), se = se(x)))
summ_B <- do.call(data.frame, summ_B)
names(summ_B) <- c("n_per_type_f", "method", "mean", "se")

g_B <- ggplot(summ_B, aes(x = n_per_type_f, y = mean, color = method, group = method)) +
    geom_line(position = position_dodge(0.15)) +
    geom_pointrange(aes(ymin = mean - se, ymax = mean + se), position = position_dodge(0.15), fatten = 1.2) +
    theme_classic() +
    ggtitle("COVID-19 cell titration") +
    xlab("Reference cells per cell type") +
    ylab("CLIMB per-sample RMSE") +
    theme(plot.title = element_text(face = "plain")) +
    scale_color_manual(values = method_palette)
g_B

# Save figure + underlying data to ../../results/panelB (relative to this repo)
out_dir_B <- "../../results/panelB"
dir.create(out_dir_B, recursive = TRUE, showWarnings = FALSE)
write.csv(df_coverage, file.path(out_dir_B, "panelB_coverage_data.csv"), row.names = FALSE)
ggsave(file.path(out_dir_B, "panelB_reference_coverage.pdf"), g_B + theme(legend.position = "none"),
       width = 5.5, height = 3.2)

## Panel C — CLIFF identifiability under correlated / low-variance cell-subtype proportions (Reviewer 2, comment #1)

*"When subtype proportions are highly correlated or show limited variability, the mixture model may suffer from identifiability issues... An explicit discussion of identifiability conditions or sensitivity analyses under correlated subtype proportions would further strengthen the framework."*

**Identifiability condition (theory):** CLIFF's E-step weights are `q_nk ∝ w_nk · π_kn` (Methods Eq. 19), and the M-step (Eq. 20) fits a weighted logistic regression where each cell type `k`'s contribution across the cohort is weighted by `γ_k = Σ_n q_nk · y_n`. Identifiability of the per-cell-type sensitivity parameters therefore depends on the **N×K matrix of proportions, W** (rows = samples, columns = cell types), not on the drug-response data: if `rank(W) < K`, the individual per-cell-type parameters are not identifiable, only certain linear combinations of them; if `W` has full rank but the cell types' across-sample proportion profiles are highly correlated or have little variance, the corresponding estimates become poorly determined (unstable), the same phenomenon as multicollinearity in ordinary regression.

**Simulation:** a purpose-built ground-truth test (no existing dataset lets us control cross-sample proportion variability directly). 4 known cell-type sensitivities (0.15, 0.35, 0.6, 0.85), cross-sample proportions drawn from a compound-symmetric log-normal (softmax of a multivariate normal with pairwise correlation `rho`), fed directly into the real, unmodified `cliff()`.

**Metric:** `cliff()` internally recenters `auc` to mean 0.5, shifting every recovered value by an unknown constant, so raw recovered-vs-true comparisons would conflate that harmless shift with real error. The natural alternative, correlation/R² (regressing recovered on true, pooling all 5 reps per `rho`), turns out to over-correct: it also cancels out *scale*, so it stays misleadingly high (R² > 0.89) even at `rho`=0.99, where the recovered values are nearly indistinguishable across cell types — a tiny but consistent trend across reps is enough for a good linear fit regardless of how compressed the values are. The metric actually used instead: **RMSE after correcting only `cliff()`'s known mean-shift** (`recovered − mean(recovered) + mean(true_sens)`, matching location but not rescaling), in the same 0–1 sensitivity units as the ground truth — so genuine range-compression still shows up as elevated error.

In [ ]:
make_identifiability_sim <- function(n_samples, k, rho, true_sens, g_per_type = 50, noise_sd = 0.03, seed = NULL) {
    if (!is.null(seed)) set.seed(seed)
    Sigma <- matrix(rho, k, k); diag(Sigma) <- 1
    x <- MASS::mvrnorm(n_samples, mu = rep(0, k), Sigma = Sigma)
    ex <- exp(x); w <- ex / rowSums(ex)
    colnames(w) <- paste0("CT", 1:k); rownames(w) <- paste0("sample", 1:n_samples)

    # True generative process matches CLIFF's own assumed mixture: bulk AUC is the
    # proportion-weighted sum of (unobserved) cell-type-specific sensitivities.
    auc <- rowSums(w * matrix(true_sens, n_samples, k, byrow = TRUE)) + rnorm(n_samples, sd = noise_sd)
    auc <- pmin(pmax(auc, 0.02), 0.98)

    # Marker-gene-block expression: each cell type has a uniquely high-expressed gene
    # block, giving CLIFF's regression something to key each cell type's identity on.
    G <- g_per_type * k
    expr_overall <- matrix(0.1, nrow = G, ncol = k)
    for (kk in 1:k) expr_overall[((kk - 1) * g_per_type + 1):(kk * g_per_type), kk] <- 10
    rownames(expr_overall) <- paste0("gene", 1:G); colnames(expr_overall) <- paste0("CT", 1:k)
    expr_highres <- array(0, dim = c(n_samples, G, k))
    for (n in 1:n_samples) expr_highres[n, , ] <- expr_overall
    dimnames(expr_highres) <- list(rownames(w), rownames(expr_overall), colnames(expr_overall))

    climb_synth <- list(props.corrected = w, expr.highres = expr_highres, expr.overall = expr_overall)
    drug_data <- data.frame(sample = rownames(w), auc = auc)
    list(climb_out = climb_synth, drug_data = drug_data, true_w = w)
}

k <- 4
true_sens <- c(0.15, 0.35, 0.6, 0.85)
rho_levels <- c(0, 0.3, 0.6, 0.8, 0.9, 0.95, 0.99)
n_reps <- 5

rows <- list()
for (rho in rho_levels) {
    for (rep in 1:n_reps) {
        d <- make_identifiability_sim(n_samples = 150, k = k, rho = rho, true_sens = true_sens, seed = rep * 100 + round(rho * 1000))
        mean_celltype_var <- mean(apply(d$true_w, 2, var))
        cliff_out <- tryCatch(cliff(d$climb_out, d$drug_data, mode = "overall", max.em.steps = 100, regularization = "none"),
                               error = function(e) NULL)
        if (is.null(cliff_out)) next
        recovered <- as.numeric(colMeans(cliff_out[[1]], na.rm = TRUE)[paste0("CT", 1:k)])
        # Correct only cliff()'s known mean-shift (location, not scale) before computing RMSE
        # against the true sensitivities — see Panel C markdown for why this is the right
        # accuracy metric here (R²/correlation-based metrics stay misleadingly high).
        recovered_debiased <- recovered - mean(recovered) + mean(true_sens)
        rows[[length(rows) + 1]] <- data.frame(rho = rho, rep = rep,
            mean_celltype_var = mean_celltype_var,
            rmse = sqrt(mean((recovered_debiased - true_sens)^2)))
    }
}
df_identifiability <- do.call(rbind, rows)
aggregate(cbind(mean_celltype_var, rmse) ~ rho, df_identifiability, mean)

In [ ]:
df_identifiability$rho_f <- factor(df_identifiability$rho, levels = rho_levels)

g_C <- ggplot(df_identifiability, aes(x = rho_f, y = rmse)) +
    geom_boxplot(outlier.shape = NA, width = 0.6, color = "cornflowerblue", fill = NA) +
    geom_jitter(width = 0.1, height = 0, size = 1.6, alpha = 0.8, color = "cornflowerblue") +
    theme_classic() +
    xlab("Cross-sample cell type\nproportion correlation (rho)") +
    ylab("CLIFF prediction error (RMSE vs. truth)")
g_C

out_dir_C <- "../../results/panelC"
dir.create(out_dir_C, recursive = TRUE, showWarnings = FALSE)
write.csv(df_identifiability, file.path(out_dir_C, "panelC_identifiability_data.csv"), row.names = FALSE)
ggsave(file.path(out_dir_C, "panelC_identifiability.pdf"), g_C, width = 5, height = 3.2)

## Panel D — EM convergence / initialization sensitivity: restart scatter (Reviewer 2, comment #2)

*"It is unclear how sensitive the results are to initialization of π or β, or whether multiple local optima exist. The manuscript would benefit from a more thorough analysis of EM convergence properties, including sensitivity to initialization and potential local optima."*

**Confirmed text/code mismatch (needs reconciling independent of this analysis):** manuscript p.25 states β̃ is optimized "through a ten-fold cross-validation (taking 90% of the samples N in the training set, and 10% in the testing set), and using early-stopping." The shipped `cliff()` does neither — no train/test split, no cross-validation anywhere (plain `glmnet()`, never `cv.glmnet()`), and early-stopping is based on full-set RMSE plateauing for 3 consecutive iterations, not held-out performance.

**The initialization question can't be answered by the shipped code as-is:** `cliff()` always sets `pi_hat = mean_auc`, a single constant, identical for every row, every run. This uses a modified copy of `cliff()` (`cliff_beta_init()`, identical body, one added parameter) that starts the EM loop from an arbitrary β coefficient vector instead. `cliff()`'s actual default start (`pi_hat = mean_auc` for every row) is mathematically identical to β = 0 exactly — the sigmoid's offset is defined so `sigmoid_(0, offset) = mean_auc` — so β = 0 is not an approximation of the shipped default, it *is* the shipped default.

**Both axes use Panel C's exact metric** (debiased RMSE of recovered vs. true per-cell-type sensitivities), not `cliff()`'s internal fit-to-observed-AUC diagnostic — an earlier version of this panel used the latter and produced numbers that looked contradictory with Panel C (e.g. ρ=0.99 appearing "better" than ρ=0.5). That was correct but easy to misread: `cliff()`'s internal RMSE measures fit to the data it was given, not accuracy against the truth, and at high ρ the simulated bulk AUCs themselves collapse toward a near-constant value (proportions become nearly identical across every sample), so *any* reasonable fit looks good by that internal measure even as the recovered cell-type-specific values become meaningless. Plotting the truth-based metric throughout removes that ambiguity — as confirmed below, the default (β=0) reference point's final RMSE lands within ~0.02 of Panel C's own value at the matching ρ, exactly as expected since it's the identical algorithm from the identical starting point.

**Restart scatter, on the same rho=0.9 correlated-proportions scenario from Panel C, repeated at rho = 0.5, 0.9, 0.99:** for each rho, 25 restarts from β ~ Normal(0, sd=60) (i.i.d. across all coefficients). X-axis: sensitivity RMSE after just the *first* EM iteration; y-axis: sensitivity RMSE at convergence; point size: iterations taken to converge; dashed crosshair per rho (own legend entry): `cliff()`'s own unmodified default start (β=0), both its starting quality (vertical line) and where it ends up (horizontal line).

**Finding:** the crosshair's height tracks Panel C exactly (0.5→0.160, 0.9→0.165, 0.99→0.261 here vs. 0.159/0.180/0.258 in Panel C). But the *restarts* diverge from that reference increasingly with ρ: at ρ=0.5, most restarts (15/25) land within 0.02 of the default and only 4/25 do worse; at ρ=0.99, *none* land near the default and 19/25 do distinctly worse (mean 0.36 vs. the default's 0.26, worst case 0.55). So poor identifiability doesn't just make the default run's recovery worse (Panel C) — it makes the *entire landscape* far more sensitive to where you start, since a flatter fit-to-data objective (many equally-good-looking fits) gives random perturbations much more room to land somewhere that fits the data fine but recovers the wrong cell-type values. Panel E continues this with a direct sweep of the starting-point distribution.

In [ ]:
# Modified copy of cliff() (verbatim body from ClimbTheCliff/R/cliff.R), with exactly one
# change: the EM is started from an arbitrary beta coefficient vector (init_beta) instead of
# the hardcoded constant pi_hat = mean_auc. Everything else — including the set.seed(1)
# before every M-step fit — is identical to the shipped function. Also tracks, at the first
# EM iteration and at convergence: first_step_rmse/min_rmse (fit to observed AUC -- the
# internal diagnostic cliff() itself uses) and first_truth_rmse/final_truth_rmse (distance to
# the known true_sens -- only computable here because this is a simulation with ground truth,
# exactly the same debiased metric used in Panel C).
cliff_beta_init <- function(climb_output, drug_data, mutation_data = NULL, min.mutation = 0,
                             max.em.steps = 100, mode = "highres", regularization = "none",
                             cancer_pattern = 'like', init_beta = NULL, true_sens = NULL) {
    sigmoid_ <- function(x, a) { return(1 / (1 + exp(-x + a))) }
    rmse_ <- function(tr, pr) { return(sqrt(sum((tr - pr)^2) / length(tr))) }
    num <- function(x) { return(as.numeric(as.character(x))) }

    climb_expr = climb_output$expr.highres
    climb_prop = climb_output$props.corrected
    rownames(climb_prop) = dimnames(climb_expr)[[1]]
    climb_expr_overall = climb_output$expr.overall
    if (is.null(mutation_data)) {
        mutation_data = matrix(0, ncol = 2, nrow = dim(climb_prop)[1])
        colnames(mutation_data) = c("a", "b"); rownames(mutation_data) = rownames(climb_prop)
    }
    mean_auc = mean(drug_data$auc)
    drug_data$auc = drug_data$auc + (0.5 - mean(mean(drug_data$auc)))
    sel.sample = Reduce(intersect, list(dimnames(climb_expr)[[1]], rownames(mutation_data), drug_data$sample))
    order.sample.climb = match(sel.sample, dimnames(climb_expr)[[1]])
    drug_data = drug_data[match(sel.sample, drug_data$sample), ]
    climb_expr = climb_expr[order.sample.climb, , ]
    climb_prop = climb_prop[order.sample.climb, ]
    mutation_data = mutation_data[sel.sample, ]
    N = dim(climb_prop)[1]
    if (sum(colSums(mutation_data) >= min.mutation) == 0) {
        mutation_data = matrix(0, ncol = 2, nrow = N)
        colnames(mutation_data) = c("a", "b"); rownames(mutation_data) = rownames(climb_prop)
    } else {
        sel.mutation = colSums(mutation_data) >= min.mutation
        mutation_data = mutation_data[, sel.mutation]
        mutation_data = as.matrix(mutation_data)
    }
    stopifnot(all(dimnames(climb_expr)[[1]] == rownames(mutation_data)))
    stopifnot(all(dimnames(climb_expr)[[1]] == drug_data$sample))
    K = num(dim(climb_expr)[3])
    N = num(dim(climb_expr)[1])
    sample_names = dimnames(climb_expr)[[1]]
    tabs_ = list()
    for (n in 1:N) {
        if (mode == "overall") {
            climb_expr[n, , ] = climb_expr_overall
        } else {
            q99 = quantile(climb_expr[n, , ], p = 0.95)
            climb_expr[n, , ][climb_expr[n, , ] > q99] <- q99
        }
        df_expr = data.frame(t(log2(climb_expr[n, , ] + 1)) / rowSums(t(log2(climb_expr[n, , ] + 1)))) * climb_prop[n, ]
        df_expr[is.na(df_expr)] <- 0
        rownames(df_expr) = paste0(rownames(df_expr), "_", sample_names[n])
        df_expr$props = climb_prop[n, ]
        mat.mut = matrix(0, ncol = length(mutation_data[n, ]), nrow = length(colnames(climb_prop)))
        mat.mut[grepl(cancer_pattern, colnames(climb_prop)), ] = mutation_data[n, ]
        colnames(mat.mut) = paste0("mut.", colnames(mutation_data))
        df_expr_0 = cbind(df_expr, mat.mut); df_expr_1 = cbind(df_expr, mat.mut)
        rownames(df_expr_0) = paste0(rownames(df_expr_0), ".0")
        rownames(df_expr_1) = paste0(rownames(df_expr_1), ".1")
        tabs_[[n]] = rbind(df_expr_0, df_expr_1)
    }
    cliff_input = do.call(rbind, tabs_)
    prop_col = num(cliff_input[, grepl("props", colnames(cliff_input))])
    cliff_input = cliff_input[, !grepl("props", colnames(cliff_input))]
    drug_data = drug_data[rep(seq_len(nrow(drug_data)), each = 2 * K), ]
    drug_data$y = drug_data$auc
    drug_data$y_bin = drug_data$auc
    for (i in 0:(2 * N - 1)) {
        start_ = i * K + 1; end_ = (i + 1) * K
        if (i %% 2 == 1) drug_data$y_bin[start_:end_] <- 0 else drug_data$y_bin[start_:end_] <- 1
    }
    # --- only change from the shipped cliff(): start the EM loop from an arbitrary beta ---
    # init_beta = NULL reproduces the shipped default exactly (pi_hat = mean_auc for every
    # row), since sigmoid_(0, offset) == mean_auc given the offset used below -- beta = 0
    # *is* the shipped default, not an approximation of it.
    offset_ <- log((1 - mean_auc) / mean_auc)
    drug_data$pi_hat = if (is.null(init_beta)) mean_auc else as.vector(sigmoid_(as.matrix(cliff_input) %*% init_beta, offset_))
    drug_data$y_hat = 0; drug_data$q_0 = 0; drug_data$q_1 = 0; drug_data$w_0 = 0; drug_data$w_1 = 0
    drug_data$y = drug_data$auc
    it.increasing.rmse = -1
    min_rmse = 1000
    min_coefs <- NULL; min_PI <- NULL
    first_step_rmse <- NA; first_truth_rmse <- NA; final_truth_rmse <- NA
    final_e <- max.em.steps
    for (e in 1:max.em.steps) {
        for (i in 0:(2 * N - 1)) {
            start_ = i * K + 1; end_ = (i + 1) * K
            div_factor_0 = sum(prop_col[start_:end_] * (1 - drug_data$pi_hat[start_:end_]))
            div_factor_1 = sum(prop_col[start_:end_] * drug_data$pi_hat[start_:end_])
            drug_data$q_0[start_:end_] <- ((prop_col[start_:end_] * (1 - drug_data$pi_hat[start_:end_])) / div_factor_0)
            drug_data$q_1[start_:end_] <- ((prop_col[start_:end_] * drug_data$pi_hat[start_:end_]) / div_factor_1)
            drug_data$w_0[start_:end_] <- (1 - drug_data$y[start_:end_]) * drug_data$q_0[start_:end_]
            drug_data$w_1[start_:end_] <- drug_data$y[start_:end_] * drug_data$q_1[start_:end_]
        }
        y = drug_data$y_bin
        drug_data$weights = drug_data$w_0
        drug_data$weights[drug_data$y_bin == 1] <- drug_data$w_1[drug_data$y_bin == 1]
        w_ = drug_data$weights
        set.seed(1)
        if (regularization == "L2") {
            fit = glmnet(as.matrix(cliff_input), y, weights = w_, family = "binomial", intercept = F, scale = F, alpha = 0)
        } else {
            fit = glmnet(as.matrix(cliff_input), y, weights = w_, family = "binomial", intercept = F, scale = F, lambda = 0)
        }
        cliff_coefs = coef(fit)[-1, dim(coef(fit))[2]]
        drug_data$pi_hat = sigmoid_(as.matrix(cliff_input) %*% cliff_coefs, offset_)
        for (i in 0:(2 * N - 1)) {
            start_ = i * K + 1; end_ = (i + 1) * K
            drug_data$y_hat[start_:end_] <- sum(prop_col[start_:end_] * drug_data$pi_hat[start_:end_])
        }
        drug_data.sub = drug_data[drug_data$y_bin == 1, ]
        drug_data.sub$cellType = rep(colnames(climb_prop), N)
        pi_hat_nk = data.frame(drug_data.sub$pi_hat)
        pi_hat_nk$sample = gsub(".*_", "", rownames(pi_hat_nk))
        pi_hat_nk$cellType = gsub("_.*", "", rownames(pi_hat_nk))
        colnames(pi_hat_nk) = c("pi_hat_nk", "sample", "celltype")
        PI_hat_nk = dcast(pi_hat_nk, sample ~ celltype, value.var = "pi_hat_nk")
        rownames(PI_hat_nk) = PI_hat_nk[, 1]; PI_hat_nk = PI_hat_nk[, -1]
        rownames(PI_hat_nk) = gsub("\\.0", "", rownames(PI_hat_nk))
        PI_hat_nk = PI_hat_nk[rownames(mutation_data), ]
        auc_values = drug_data.sub$auc[seq(1, length(drug_data.sub$auc), K)]
        rmse_val = rmse_(rowSums(as.matrix(PI_hat_nk) * as.matrix(climb_prop)), auc_values)
        # Ground-truth accuracy at this iteration (only computable in simulation, where
        # true_sens is known) -- same debiasing as Panel C: cliff() recenters auc to 0.5
        # internally, which shifts every recovered value by an unknown constant, so we
        # match location (not scale) before comparing to the truth.
        truth_rmse_e <- NA
        if (!is.null(true_sens)) {
            recovered_e <- colMeans(as.matrix(PI_hat_nk), na.rm = TRUE)
            recovered_e_debiased <- recovered_e - mean(recovered_e) + mean(true_sens)
            truth_rmse_e <- sqrt(mean((recovered_e_debiased - true_sens)^2))
        }
        if (e == 1) { first_step_rmse <- rmse_val; first_truth_rmse <- truth_rmse_e }
        if (e > 1 & rmse_val < min_rmse & abs(rmse_val - min_rmse) > 1e-3) {
            min_rmse = rmse_val; min_coefs = cliff_coefs; min_PI = PI_hat_nk
            final_truth_rmse <- truth_rmse_e
            it.increasing.rmse = 0
        } else {
            it.increasing.rmse = it.increasing.rmse + 1
            if (it.increasing.rmse >= 3) { final_e <- e; break }
        }
    }
    list(PI_hat_nk = min_PI, min_rmse = min_rmse, min_coefs = min_coefs, n_iters = final_e,
         first_step_rmse = first_step_rmse, first_truth_rmse = first_truth_rmse, final_truth_rmse = final_truth_rmse)
}

In [ ]:
# --- Part 1: restart scatter across 3 rho values ---
rho_levels_D <- c(0.5, 0.9, 0.99)
n_restarts_D <- 25
sd_scatter <- 60  # sd of the i.i.d. Normal(0, sd) draw for each restart's starting beta vector

set.seed(2)
d_list_D <- lapply(rho_levels_D, function(rho)
    make_identifiability_sim(n_samples = 150, k = k, rho = rho, true_sens = true_sens, seed = 777))
ncol_beta_D <- dim(d_list_D[[1]]$climb_out$expr.highres)[2] + 2  # marker genes + 2 always-zero mutation columns

# All random beta draws generated upfront in one block: cliff_beta_init() calls set.seed(1)
# internally before every M-step glmnet fit, which would otherwise silently reset this
# loop's RNG mid-sweep.
beta_draws_scatter <- array(rnorm(length(rho_levels_D) * n_restarts_D * ncol_beta_D, mean = 0, sd = sd_scatter),
                             dim = c(length(rho_levels_D), n_restarts_D, ncol_beta_D))

scatter_rows <- list(); default_rows <- list()
for (ri in seq_along(rho_levels_D)) {
    rho <- rho_levels_D[ri]; d_r <- d_list_D[[ri]]
    # beta = 0 reproduces cliff()'s actual unmodified default start exactly (see Panel D
    # markdown) -- one reference point per rho, used for the dashed crosshair below. Its
    # final_truth_rmse is expected to land close to Panel C's curve at the same rho (same
    # algorithm, same starting point, same debiased truth metric).
    default_out <- cliff_beta_init(d_r$climb_out, d_r$drug_data, mode = "overall", max.em.steps = 100,
                                    init_beta = rep(0, ncol_beta_D), true_sens = true_sens)
    default_rows[[ri]] <- data.frame(rho = rho, first_step_rmse = default_out$first_step_rmse,
                                      min_rmse = default_out$min_rmse,
                                      first_truth_rmse = default_out$first_truth_rmse,
                                      final_truth_rmse = default_out$final_truth_rmse)
    for (i in 1:n_restarts_D) {
        out <- cliff_beta_init(d_r$climb_out, d_r$drug_data, mode = "overall", max.em.steps = 100,
                                init_beta = beta_draws_scatter[ri, i, ], true_sens = true_sens)
        scatter_rows[[length(scatter_rows) + 1]] <- data.frame(rho = rho, restart = i,
            first_step_rmse = out$first_step_rmse, min_rmse = out$min_rmse, n_iters = out$n_iters,
            first_truth_rmse = out$first_truth_rmse, final_truth_rmse = out$final_truth_rmse)
    }
}
df_em <- do.call(rbind, scatter_rows)
df_em_default <- do.call(rbind, default_rows)
df_em_default

# --- Part 2: initialization-landscape heatmap (beta average x sd grid), single reference
# scenario (rho = 0.9, same as used throughout Panels C-D) ---
avg_grid <- seq(-20, 20, by = 10)
sd_grid  <- c(0, 20, 40, 60, 80, 100)
n_reps_heat <- 3

d_heat <- d_list_D[[which(rho_levels_D == 0.9)]]
ncol_beta_heat <- dim(d_heat$climb_out$expr.highres)[2] + 2

set.seed(3)
grid_specs <- list(); beta_pool <- list(); idx <- 1
for (avg in avg_grid) {
    for (sdv in sd_grid) {
        reps <- if (sdv == 0) 1 else n_reps_heat  # sd=0 is deterministic given avg -- 1 rep suffices
        beta_pool[[idx]] <- matrix(rnorm(reps * ncol_beta_heat, mean = avg, sd = sdv), nrow = reps)
        grid_specs[[idx]] <- list(avg = avg, sd = sdv, reps = reps)
        idx <- idx + 1
    }
}

heat_rows <- list()
for (i in seq_along(grid_specs)) {
    spec <- grid_specs[[i]]; betas <- beta_pool[[i]]
    truth_rmses <- sapply(1:spec$reps, function(r)
        cliff_beta_init(d_heat$climb_out, d_heat$drug_data, mode = "overall", max.em.steps = 100,
                         init_beta = betas[r, ], true_sens = true_sens)$final_truth_rmse)
    heat_rows[[i]] <- data.frame(beta_avg = spec$avg, beta_sd = spec$sd, mean_truth_rmse = mean(truth_rmses), n_reps = spec$reps)
}
df_em_heat <- do.call(rbind, heat_rows)
df_em_heat

In [ ]:
# NB: Unicode rho/beta glyphs don't render in this R's pdf() device (show up as missing-
# glyph placeholders) -- spelled out as "rho"/"beta" in all labels below for portability.
rho_labels <- paste0("rho = ", rho_levels_D)
df_em$rho_f <- factor(paste0("rho = ", df_em$rho), levels = rho_labels)
df_em_default$rho_f <- factor(paste0("rho = ", df_em_default$rho), levels = rho_labels)
rho_palette <- setNames(RColorBrewer::brewer.pal(3, "Set2"), rho_labels)

# Both axes are now the same debiased truth-vs-recovered RMSE Panel C uses (not the
# internal fit-to-observed-AUC RMSE cliff() uses for its own early stopping) -- x = quality
# of the starting point after 1 EM iteration, y = quality at convergence, size = iterations
# to converge. The dashed crosshair per rho marks cliff()'s own unmodified default start
# (beta=0), labeled via its own legend entry (separate from the rho color legend, which the
# crosshair also inherits so each line is color-matched to its rho).
g_D <- ggplot(df_em, aes(x = first_truth_rmse, y = final_truth_rmse, color = rho_f, size = n_iters)) +
    geom_point(alpha = 0.75) +
    geom_vline(data = df_em_default, aes(xintercept = first_truth_rmse, color = rho_f, linetype = "Default init (all beta = 0)"),
               linewidth = 0.6) +
    geom_hline(data = df_em_default, aes(yintercept = final_truth_rmse, color = rho_f, linetype = "Default init (all beta = 0)"),
               linewidth = 0.6) +
    scale_color_manual(values = rho_palette, name = NULL) +
    scale_linetype_manual(name = NULL, values = c("Default init (all beta = 0)" = "dashed")) +
    guides(linetype = guide_legend(override.aes = list(color = "black"))) +
    theme_classic() +
    xlab("EM first iteration error (RMSE vs. truth)") +
    ylab("CLIFF prediction error (RMSE vs. truth)") +
    labs(size = "EM iterations\nto converge")
g_D

out_dir_D <- "../../results/panelD"
dir.create(out_dir_D, recursive = TRUE, showWarnings = FALSE)
write.csv(df_em, file.path(out_dir_D, "panelD_em_restarts_data.csv"), row.names = FALSE)
write.csv(df_em_default, file.path(out_dir_D, "panelD_em_default_by_rho.csv"), row.names = FALSE)
ggsave(file.path(out_dir_D, "panelD_scatter.pdf"), g_D, width = 6, height = 3.6)

## Panel E — EM convergence / initialization sensitivity: initialization landscape (Reviewer 2, comment #2, continued)

Direct sweep over the initial β distribution used to start the EM loop (same `cliff_beta_init()` and same rho=0.9 reference scenario as Panel C/D): x-axis = mean of the starting β coefficients (-20 to +20), y-axis = their standard deviation (0 to 100), cell value/color = mean final truth-based RMSE (Panel C's metric) across 3 restarts per cell (1 for sd=0, which is deterministic). The black-outlined cell (average=0, sd=0) is `cliff()`'s actual unmodified default start (RMSE 0.165, matching Panel D's crosshair).

**Finding:** the *average* of the starting coefficients has little effect at low spread (the sd=0 row stays ~0.16-0.17 throughout, regardless of average). What matters is the *spread*: once sd exceeds ~60-80, accuracy degrades sharply and unevenly (up to 0.42), regardless of where it's centered — consistent with Panel D's restarts, which used sd=60.

**Mitigation, unchanged from before:** picking the best (lowest first-EM-iteration or final RMSE against the *fit-to-data* diagnostic — the only thing available without ground truth) of a handful of random restarts is a concrete, practical fix that needs no access to ground truth, though at high ρ (Panel D) this shows that mitigation has less to work with: many restarts fit the data equally well while recovering very different (and mostly wrong) cell-type values.

In [ ]:
# Sweep of the starting beta distribution's average (x) and standard deviation (y), colored
# by mean FINAL TRUTH RMSE (same metric as Panel C/D, not fit-to-data). The black-outlined
# cell (average=0, sd=0) is cliff()'s actual unmodified default start.
mid_rmse <- mean(range(df_em_heat$mean_truth_rmse))
g_E <- ggplot(df_em_heat, aes(x = factor(beta_avg), y = factor(beta_sd), fill = mean_truth_rmse)) +
    geom_tile() +
    geom_text(aes(label = sprintf("%.3f", mean_truth_rmse), color = mean_truth_rmse > mid_rmse), size = 2.6, show.legend = FALSE) +
    geom_tile(data = subset(df_em_heat, beta_avg == 0 & beta_sd == 0), fill = NA, color = "black", linewidth = 1) +
    scale_fill_gradient(low = "grey95", high = "firebrick", name = "CLIFF prediction\nerror (RMSE\nvs. truth)") +
    scale_color_manual(values = c("TRUE" = "white", "FALSE" = "black")) +
    theme_classic() +
    xlab("Initial beta average") + ylab("Initial beta standard deviation") +
    ggtitle("black box = cliff()'s actual default start (beta=0)") +
    theme(plot.title = element_text(size = 8, face = "italic"))
g_E

out_dir_E <- "../../results/panelE"
dir.create(out_dir_E, recursive = TRUE, showWarnings = FALSE)
write.csv(df_em_heat, file.path(out_dir_E, "panelE_heatmap_data.csv"), row.names = FALSE)
ggsave(file.path(out_dir_E, "panelE_heatmap.pdf"), g_E, width = 5.5, height = 3.6)

**Next steps for this panel:**
- ~~Decide whether to use in-vitro or a non-hematopoietic dataset~~ — decided: COVID-19 nasal-swab, both non-hematopoietic (matching W5's wording) and giving a fairer NNLS comparison.
- The NNLS baseline here (like Panel A's) uses all common genes with no marker/variable-gene selection — same caveat as Panel A.

## Panel G — Uncertainty propagation from CLIMB into CLIFF (Reviewer 2, comment #4)

*"Please clarify how uncertainty in cell-type proportion estimates from CLIMB propagates into the inference of cell-type-specific drug sensitivities in CLIFF."*

**First attempt (not shown, didn't work):** running `climb()`/NNLS directly on a synthetic single-cell reference to get real `props.init`/`props.corrected`/NNLS estimates. A clean, extreme-fold-change marker-gene simulation makes simple averaging (NNLS) *nearly noiseless* — there's no real cell-to-cell heterogeneity for CLIMB's single-cell resolution to exploit, so `props.corrected` came out worse than NNLS (prop RMSE 0.26 vs. 0.0009), the reverse of the real result in Panel A. Not something to show as-is.

**Final design — calibrated noise injection:** starting from the same rho=0.3 identifiability scenario (well-identified regime, distinct from Panel C's rho sweep), inject controlled multiplicative log-normal noise into the true proportions, calibrated (via a separate noise-vs-error calibration pass) so that its *mean proportion RMSE exactly matches each real method's actual per-sample proportion accuracy from Panel A* (0.077 / 0.097 / 0.180 for corrected / init / NNLS — the calibration lands within 0.002-0.005 of Panel A's real 0.076 / 0.095 / 0.175). The resulting noisy proportion matrix is substituted for `props.corrected` in `cliff()`'s input (same substitution mechanism as the failed attempt, just driven by a controlled, calibrated noise source instead of literally re-running CLIMB/NNLS on a toy reference). 10 reps per source; error bars are SE.

Four points, same color scheme as Panels A/B (NNLS/init/corrected) plus cornflowerblue for the true, error-free proportions (error = 0 by construction). The light grey points/dashed trend in the background are the same mechanism run continuously across 7 noise levels, showing the full dose-response — the "what if we swept noise continuously" extension.

**Finding:** a clean, monotonic degradation matching Panel A's own method ranking — CLIFF sensitivity RMSE: True proportions 0.158 -> CLIMB corrected 0.185 -> CLIMB init 0.194 -> NNLS 0.231. Even with perfect (true) proportions, CLIFF's error doesn't reach zero — an irreducible floor from AUC observation noise and finite sample size alone. This directly answers the reviewer: CLIMB's proportion uncertainty propagates into CLIFF roughly linearly across the accuracy range real methods actually achieve, and the *real, measured* accuracy gap between methods (corrected vs. NNLS, a factor of ~2.3x in proportion RMSE) translates into a real but more modest ~25% gap in downstream sensitivity accuracy.

**Included as the last panel (G) of the combined supplemental figure**, alongside its own standalone PDF.

In [ ]:
# Well-identified baseline (rho=0.3, distinct from Panel C's rho sweep) so this panel
# isolates the propagation question from Panel C's identifiability question.
d_prop <- make_identifiability_sim(n_samples = 150, k = k, rho = 0.3, true_sens = true_sens, seed = 999)
true_w <- d_prop$true_w

run_cliff_rmse <- function(prop_matrix) {
    climb_variant <- d_prop$climb_out
    climb_variant$props.corrected <- prop_matrix
    cliff_out <- cliff(climb_variant, d_prop$drug_data, mode = "overall", max.em.steps = 100, regularization = "none")
    recovered <- as.numeric(colMeans(cliff_out[[1]], na.rm = TRUE)[paste0("CT", 1:k)])
    recovered_debiased <- recovered - mean(recovered) + mean(true_sens)
    sqrt(mean((recovered_debiased - true_sens)^2))
}

# noise_sd calibrated (separately, via a bisection-style sweep) so that the mean proportion
# RMSE of the resulting noisy matrix matches each REAL method's actual per-sample proportion
# RMSE from Panel A's cross-dataset results (0.076 / 0.095 / 0.175 for corrected/init/NNLS).
noise_by_method <- c("Single-cell, empirical-Bayes corrected (props.corrected)" = 0.38,
                      "Single-cell, uncorrected (props.init)" = 0.48,
                      "Cell-type aggregated (NNLS)" = 0.97)
n_reps_prop <- 10

prop_rows <- list()
for (rep in 1:n_reps_prop) {
    set.seed(rep * 111)
    prop_rows[[length(prop_rows) + 1]] <- data.frame(source = "True cell-type proportion",
        prop_rmse = 0, cliff_rmse = run_cliff_rmse(true_w))
}
for (method in names(noise_by_method)) {
    noise_sd <- noise_by_method[method]
    for (rep in 1:n_reps_prop) {
        set.seed(rep * 777 + round(noise_sd * 10000))
        noisy_w <- true_w * matrix(exp(rnorm(length(true_w), sd = noise_sd)), nrow(true_w), ncol(true_w))
        noisy_w <- noisy_w / rowSums(noisy_w)
        prop_rmse <- sqrt(mean((noisy_w - true_w)^2))
        prop_rows[[length(prop_rows) + 1]] <- data.frame(source = method,
            prop_rmse = prop_rmse, cliff_rmse = run_cliff_rmse(noisy_w))
    }
}
df_g <- do.call(rbind, prop_rows)

# Light background context: same mechanism, swept across a finer grid of noise levels
# (not calibrated to any specific method), 4 reps per level AVERAGED to one point each
# (rather than plotting all 4 raw repeats) for a clean, smooth curve.
noise_levels <- c(0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.65, 0.8, 1.0, 1.2, 1.4, 1.6)
n_reps_curve <- 4
curve_rows <- list()
for (noise_sd in noise_levels) {
    for (rep in 1:n_reps_curve) {
        set.seed(rep * 1000 + round(noise_sd * 1000))
        noisy_w <- if (noise_sd == 0) true_w else {
            nw <- true_w * matrix(exp(rnorm(length(true_w), sd = noise_sd)), nrow(true_w), ncol(true_w))
            nw / rowSums(nw)
        }
        curve_rows[[length(curve_rows) + 1]] <- data.frame(noise_sd = noise_sd,
            prop_rmse = sqrt(mean((noisy_w - true_w)^2)), cliff_rmse = run_cliff_rmse(noisy_w))
    }
}
df_curve_raw <- do.call(rbind, curve_rows)
df_curve <- aggregate(cbind(prop_rmse, cliff_rmse) ~ noise_sd, df_curve_raw, mean)

# Noise expressed as % of signal: the coefficient of variation of the multiplicative
# log-normal factor exp(N(0, noise_sd)) applied to each true proportion, CV = sqrt(exp(sd^2)-1).
df_curve$noise_pct <- sqrt(exp(df_curve$noise_sd^2) - 1) * 100
df_g

In [ ]:
suppressMessages(library(ggnewscale))  # lets the background noise gradient and the 4 main
                                        # points use two independent color scales

source_labels <- c("True cell-type proportion" = "True cell-type proportion",
                    "Single-cell, empirical-Bayes corrected (props.corrected)" = "CLIMB (corrected)",
                    "Single-cell, uncorrected (props.init)" = "CLIMB (init, uncorrected)",
                    "Cell-type aggregated (NNLS)" = "NNLS")
source_order <- c("True cell-type proportion", "CLIMB (corrected)", "CLIMB (init, uncorrected)", "NNLS")
source_palette <- c("True cell-type proportion" = "cornflowerblue",
                     setNames(method_palette, source_labels[names(method_palette)]))

df_g$source_label <- factor(source_labels[df_g$source], levels = source_order)
summ_g <- aggregate(cbind(prop_rmse, cliff_rmse) ~ source_label, df_g, mean)
se_g <- aggregate(cliff_rmse ~ source_label, df_g, function(x) sd(x) / sqrt(length(x)))
names(se_g)[2] <- "se"
summ_g <- merge(summ_g, se_g)

# 4 clean points (mean +/- SE, calibrated to each method's real proportion accuracy) in
# front; the continuous noise sweep (one averaged point per level) behind, colored by
# injected noise expressed as % of signal (own legend/scale via ggnewscale, placed below
# the categorical legend via guide order, same text size as the categorical legend).
g_G <- ggplot() +
    geom_point(data = df_curve, aes(x = prop_rmse, y = cliff_rmse, color = noise_pct), size = 2) +
    scale_color_gradient(low = "#c6dbef", high = "#08306b",
                          name = "Injected noise to\ntrue cell-type prop\n(% of signal)",
                          guide = guide_colorbar(order = 2)) +
    new_scale_color() +
    geom_pointrange(data = summ_g, aes(x = prop_rmse, y = cliff_rmse, ymin = cliff_rmse - se, ymax = cliff_rmse + se,
                                        color = source_label), size = 0.9, fatten = 3, linewidth = 1) +
    scale_color_manual(values = source_palette, name = NULL, guide = guide_legend(order = 1)) +
    theme_classic() +
    theme(legend.title = element_text(size = 11), legend.text = element_text(size = 10)) +
    xlab("Cell-type proportion error (RMSE vs. truth)") +
    ylab("CLIFF prediction error (RMSE vs. truth)")
g_G

out_dir_G <- "../../results/panelG"
dir.create(out_dir_G, recursive = TRUE, showWarnings = FALSE)
write.csv(df_g, file.path(out_dir_G, "panelG_propagation_data.csv"), row.names = FALSE)
write.csv(df_curve, file.path(out_dir_G, "panelG_noise_curve_data.csv"), row.names = FALSE)
ggsave(file.path(out_dir_G, "panelG_propagation.pdf"), g_G, width = 7.2, height = 4.3)

## Panel F — Runtime / computational complexity (Reviewer 2, comment #3)

*"How is the performance of CLIMB–CLIFF on large datasets? It's better to provide computational complexity or runtime benchmarks."*

Confirmed absent from the manuscript's main text. The shared `make_synthetic()` generator, timing `climb()` and `cliff()` independently while varying one axis at a time — bulk samples (N), genes (G), reference cell types (K), and total reference cells (up to 15000, at a beatAML-like scale of k=19/g=2000) — a 2x2 grid, both CLIMB and CLIFF lines on every facet, all reporting the same quantity (total wall-clock seconds for that run) so the four panels are directly comparable.

**Attempted first with real data:** subsampling the real Van Galen AML single-cell reference (`data/pseudobulks_climb/AML_vangalen_sc_es.RDS`, 15255 cells, 19 cell types) to build a "simulated beatAML"-scale benchmark — real reference, synthetic pseudobulks (cells drawn with replacement from the reference per random Dirichlet-like proportions, following the same approach used to build `data/simulation_aml`). `climb()` succeeds on the **full, untouched** reference (~50s/sample) but throws an internal array-dimension error (`non-conformable arrays`, in the expression-prediction step) on **any** subsample of it, uniform or proportional, tested from 10 to 500 cells/type — a real, reproducible package limitation worth its own investigation, out of scope for a quick benchmark. Flagged as a follow-up, not fixed here.

*(An earlier version of this panel also included a CLIFF runtime heatmap of bulk samples x reference cells — dropped once the "total reference cells" facet above made it redundant: both showed the same conclusion, CLIFF's runtime tracks N samples and is flat across reference size, and the 2x2 grid already makes that comparison directly against CLIMB.)*

In [ ]:
## --- Original sweep: N samples, N genes, N cell types, one axis at a time ---
time_climb_cliff <- function(d) {
    t0 <- Sys.time()
    co <- climb(d$sc, d$bulk, mode = "expression", verbose = FALSE, cancer_pattern = "none")
    t_climb <- as.numeric(Sys.time() - t0, units = "secs")
    t1 <- Sys.time()
    invisible(cliff(co, d$drug_data, mode = "overall", max.em.steps = 50))
    t_cliff <- as.numeric(Sys.time() - t1, units = "secs")
    c(climb = t_climb, cliff = t_cliff)
}

rows <- list()
for (n in c(10, 25, 50, 100)) {
    tt <- time_climb_cliff(make_synthetic(n, 6, 500, seed = 1))
    rows[[length(rows) + 1]] <- data.frame(axis = "n_samples", value = n, component = names(tt), seconds = tt)
}
for (g in c(200, 500, 1000, 2000)) {
    tt <- time_climb_cliff(make_synthetic(20, 6, g, seed = 1))
    rows[[length(rows) + 1]] <- data.frame(axis = "n_genes", value = g, component = names(tt), seconds = tt)
}
for (kk in c(3, 6, 10, 15)) {
    tt <- time_climb_cliff(make_synthetic(20, kk, 500, seed = 1))
    rows[[length(rows) + 1]] <- data.frame(axis = "n_celltypes", value = kk, component = names(tt), seconds = tt)
}

## --- 4th axis: TOTAL reference cells (up to 15k, beatAML scale k=19/g=2000), fixed at 10
## bulk samples. Reports the SAME quantity (raw total wall-clock seconds for that run) as
## the other 3 axes -- no per-sample normalization, so all 4 facets stay comparable.
k_panelF <- 19; g_panelF <- 2000
n_samples_ref <- 10
total_cells_grid <- c(1000, 2500, 5000, 10000, 15000)
for (total_cells in total_cells_grid) {
    ncell <- round(total_cells / k_panelF)
    tt <- time_climb_cliff(make_synthetic(n_samples_ref, k_panelF, g_panelF, n_cells_per_type = ncell, seed = 1))
    rows[[length(rows) + 1]] <- data.frame(axis = "n_cells_total", value = total_cells, component = names(tt), seconds = tt)
}
df_runtime <- do.call(rbind, rows)
rownames(df_runtime) <- NULL
df_runtime

In [ ]:
axis_labels <- c(n_samples = "Bulk samples (N)", n_genes = "Genes (G)",
                  n_celltypes = "Reference cell types (K)", n_cells_total = "Total reference cells")
df_runtime$axis_label <- factor(axis_labels[df_runtime$axis], levels = axis_labels)
component_palette <- setNames(RColorBrewer::brewer.pal(3, "Dark2")[1:2], c("climb", "cliff"))

# 2x2 grid: the 3 original axes + the total-reference-cells axis (up to 15k, beatAML
# scale), all showing the same quantity (total wall-clock seconds for that run) so the 4
# panels are directly comparable -- both CLIMB and CLIFF lines on every facet.
g_F <- ggplot(df_runtime, aes(x = value, y = seconds, color = component)) +
    geom_line() + geom_point(size = 1.2) +
    facet_wrap(~axis_label, scales = "free_x", ncol = 2) +
    theme_classic() + xlab(NULL) + ylab("Runtime (seconds)") +
    scale_color_manual(values = component_palette)
g_F

out_dir_F <- "../../results/panelF"
dir.create(out_dir_F, recursive = TRUE, showWarnings = FALSE)
write.csv(df_runtime, file.path(out_dir_F, "panelF_axis_sweep_data.csv"), row.names = FALSE)
ggsave(file.path(out_dir_F, "panelF_runtime.pdf"), g_F, width = 7, height = 6)

**Combined worst case (used in the manuscript text):** the four sweeps above each vary
one axis while holding the others fixed at small defaults, so the manuscript's claim of
"under 15 seconds at 100 samples, 15,000 reference cells" was originally a splice of two
different sweeps' endpoints (100 samples at only ~1,200 reference cells, and 15,000
reference cells at only 10 samples) rather than a single run of both together. This cell
runs the actual combined worst case -- 100 samples *and* 15,000 reference cells at the same
k=19/g=2000 beatAML-like scale as the reference-size sweep, with the same
`max.em.steps = 50` used throughout this panel -- to get a real number for the text.

In [ ]:
k_worst <- 19; g_worst <- 2000
n_samples_worst <- 100
total_cells_worst <- 15000
ncell_worst <- round(total_cells_worst / k_worst)

d_worst <- make_synthetic(n_samples_worst, k_worst, g_worst, n_cells_per_type = ncell_worst, seed = 1)
tt_worst <- time_climb_cliff(d_worst)
cat(sprintf("climb(): %.1fs, cliff(): %.1fs, combined: %.1fs (n=%d samples, %d reference cells)\n",
            tt_worst["climb"], tt_worst["cliff"], sum(tt_worst), n_samples_worst, ncell_worst * k_worst))

## Combining panels into the supplemental figure

Every panel above saves its own standalone PDF under `results/panel<X>/`. This section assembles them into the actual multi-panel supplemental figure via `cowplot::plot_grid`.

Layout:
- **Row 1** — Panel A, Panel B, and their shared method legend stacked vertically to the right of B (one shared legend since A and B use the identical 3-way method comparison; not repeated per panel). Given much less height than the other rows.
- **Row 2** — Panel C, Panel D, Panel E together (identifiability + EM initialization sensitivity, all sharing the same rho=0.9 reference scenario).
- **Row 3** — Panel F (runtime), spanning the full width.

**This section needs to be re-run/extended every time a new panel is added or a panel's plot object changes.**

In [ ]:
out_dir_combined <- "../../results"
dir.create(out_dir_combined, recursive = TRUE, showWarnings = FALSE)

# Extra bottom margin on A: its angled dataset-name tick labels otherwise overflow into
# row2's space when rows are stacked via plot_grid (cowplot doesn't clip between cells).
# Extra bottom margin on B: makes it slightly shorter than A within the same row (a plain
# rel_heights split isn't possible since A and B share one row).
# Extra top margin on C/D/E/F/G: their own labels ("C", "D", ...) otherwise sit too close
# to the row above (G's is larger since its y-axis title is long enough to reach the label).
# The A/B shared legend gets its "props.corrected" entry wrapped onto 2 lines via a
# label override applied only when extracting this legend (doesn't affect g_A itself).
legend_AB <- get_legend(g_A + theme(legend.position = "right", legend.title = element_blank()) +
                         guides(color = guide_legend(ncol = 1)) +
                         scale_color_manual(values = method_palette,
                             labels = function(x) gsub("Single-cell, empirical-Bayes corrected",
                                                        "Single-cell, empirical-Bayes\ncorrected", x)))
row1 <- plot_grid(g_A + theme(legend.position = "none", axis.text.x = element_text(angle = 35, hjust = 1, size = 7.5),
                               plot.margin = margin(t = 5.5, r = 5.5, b = 30, l = 5.5)),
                   g_B + theme(legend.position = "none", plot.margin = margin(t = 5.5, r = 5.5, b = 20, l = 5.5)),
                   legend_AB,
                   ncol = 3, labels = c("A", "B", ""), rel_widths = c(2.0, 1, 0.9))

row2 <- plot_grid(g_C + theme(plot.margin = margin(t = 14, r = 5.5, b = 5.5, l = 5.5)),
                   g_D + theme(plot.margin = margin(t = 14, r = 5.5, b = 5.5, l = 5.5)),
                   g_E + theme(plot.margin = margin(t = 14, r = 5.5, b = 5.5, l = 5.5)),
                   ncol = 3, labels = c("C", "D", "E"), rel_widths = c(0.7, 1.2, 1.05))

row3 <- plot_grid(g_F + theme(plot.margin = margin(t = 25, r = 5.5, b = 5.5, l = 5.5)),
                   g_G + theme(plot.margin = margin(t = 25, r = 5.5, b = 5.5, l = 5.5)),
                   ncol = 2, labels = c("F", "G"), rel_widths = c(1, 1))

g_supp <- plot_grid(row1, row2, row3, ncol = 1, rel_heights = c(1.6, 1.3, 1.15))

options(repr.plot.width = 13, repr.plot.height = 11.2)
g_supp

ggsave(file.path(out_dir_combined, "Supplemental_Figure_revision.pdf"), g_supp, width = 13, height = 11.2)